# Phase 2 Mock 状态推进验证

这个 notebook 用来验证当前 Phase 2 已经具备的最小状态推进能力：

- 创建 session 时会初始化 scene / objective / flags / clocks
- 提交一轮行动后会推进 scene
- objective、已发现信息、时钟和 replay 会同步更新


In [ ]:
# 这个代码块用于准备路径、端口和 HTTP 工具函数，后面的所有验证都会复用它们。
from pathlib import Path
import json
import os
import subprocess
import time
import urllib.request

repo_root = Path(r"f:\Documents\GitHub\AI_TRPG_616\version 3.0")
server_port = 4319
server_base_url = f"http://127.0.0.1:{server_port}"
server_process = None

def http_get_json(url: str) -> dict:
    # 这个辅助函数用于 GET 一个 JSON 接口。
    with urllib.request.urlopen(url) as response:
        return json.loads(response.read().decode("utf-8"))

def http_post_json(url: str, payload: dict) -> dict:
    # 这个辅助函数用于 POST 一个 JSON 接口，并显式使用 UTF-8 发送中文输入。
    request = urllib.request.Request(
        url,
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))


In [ ]:
# 这个代码块启动本地 Node 服务，并等待健康检查通过。
env = os.environ.copy()
env["PORT"] = str(server_port)

server_process = subprocess.Popen(
    ["node", "--experimental-strip-types", ".\\apps\\server\\src\\server.ts"],
    cwd=repo_root,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
)

health = None
for _ in range(20):
    try:
        health = http_get_json(f"{server_base_url}/api/health")
        break
    except Exception:
        time.sleep(0.5)

assert health is not None, "服务启动失败，健康检查没有通过"
print("健康检查结果:", health)


In [ ]:
# 这个代码块创建一个新 session，并检查初始状态是否已经包含 scene / objective / clocks。
create_payload = {
    "ruleDirectoryName": "VHS",
    "storyDirectoryName": "The_Silence",
    "locale": "zh-CN",
    "playMode": "single_player",
    "gmArchitecture": "single_agent",
    "modelAccessMode": "mock",
    "debugEnabled": True,
    "promptDebugEnabled": False,
    "logViewMode": "compact",
}

created = http_post_json(f"{server_base_url}/api/sessions", create_payload)
session_id = created["session"]["id"]
game_state = created["session"]["gameState"]

print("初始 scene:", game_state["sceneId"])
print("初始目标:", game_state["objectiveState"])
print("初始 clocks:", game_state["clocks"])
print("初始 discoveredInfoIds:", game_state["discoveredInfoIds"])

assert game_state["sceneId"] == "entry_plaza"
assert game_state["objectiveState"]["active"] == ["stabilize_entry"]
assert game_state["clocks"]["nightfall"] == 0
assert "entry_first_impression" in game_state["discoveredInfoIds"]


In [ ]:
# 这个代码块提交一轮偏向录像厅的输入，验证 scene 和 objective 会随之推进。
turn_payload = {
    "playerInput": "I head to the video hall to inspect the projector and the black tape."
}

turned = http_post_json(f"{server_base_url}/api/sessions/{session_id}/turns", turn_payload)
game_state_after_turn = turned["session"]["gameState"]
last_message = turned["messages"][-1]

print("推进后 scene:", game_state_after_turn["sceneId"])
print("推进后目标:", game_state_after_turn["objectiveState"])
print("推进后 clocks:", game_state_after_turn["clocks"])
print("推进后 discoveredInfoIds:", game_state_after_turn["discoveredInfoIds"])
print("\n最后一条主持文本:\n")
print(last_message["content"])

assert turned["session"]["currentRound"] == 1
assert game_state_after_turn["sceneId"] == "stardust_video_hall"
assert game_state_after_turn["objectiveState"]["active"] == ["inspect_video_hall"]
assert "stabilize_entry" in game_state_after_turn["objectiveState"]["completed"]
assert game_state_after_turn["clocks"]["nightfall"] == 1
assert "black_tape_hint" in game_state_after_turn["discoveredInfoIds"]


In [ ]:
# 这个代码块再次读取 session，确认 replay 里已经出现 state_patch_applied，说明状态推进被正式记录了。
fetched = http_get_json(f"{server_base_url}/api/sessions/{session_id}")
replay_types = [item["type"] for item in fetched["replay"]]
story_flags = fetched["session"]["gameState"]["storyFlags"]

print("回放类型:", replay_types)
print("关键 flags:", story_flags)

assert "state_patch_applied" in replay_types
assert story_flags["danger_level"] == "low"
assert story_flags["last_scene_title"] == "星尘录像厅"


In [ ]:
# 这个代码块用于停止后台 Node 服务，建议在验证结束后执行一次。
if server_process is not None and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_process.kill()

print("服务已停止")
